# Dwarf Example 02: DDO154 Omega Computation

**EPS Research — Dwarf/Irregular HI Corpus v1.0**

DDO154 is one of the best-studied dark-matter-dominated dwarfs.
This example reproduces the DDO154 omega computation from Flynn (2026):
omega = 6.86 rad/Gyr (SPARC rotmod) — consistent with the SPARC mean of 7.06.

The dwarf corpus median omega = 9.94 rad/Gyr (24 omega-ready LITTLE THINGS),
higher than the SPARC mean, consistent with deeper dark matter dominance.

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.20320362  
**Sources:** LVHIS (Koribalski 2019), VLA-ANGST (Ott 2012), LITTLE THINGS (Oh 2015), WALLABY DR2  
**Dependencies:** Python 3, numpy, matplotlib

In [1]:
import json
import numpy as np
import matplotlib.pyplot as plt

with open('dwarf_irregular_corpus_v1.json') as f:
    corpus = json.load(f)

# Find omega-ready dwarfs
omega_ready = [g for g in corpus['galaxies']
               if g.get('omega_ready') and g.get('data') and len(g['data']) >= 2]
print(f"Omega-ready dwarfs: {len(omega_ready)}")

# Compute omega for all
results = []
for g in omega_ready:
    d  = g['data']
    R  = [p['Rad']  for p in d]
    V  = [p['Vobs'] for p in d]
    R1, V1 = R[0],  V[0]
    R2, V2 = R[-1], V[-1]
    if R1>0 and R2>0 and V1>0 and V2>0:
        omega = (V2/R2 - V1/R1) * (R1/R2)**1.5
        results.append({'galaxy': g['galaxy'], 'omega': omega,
                        'distance': g['distance_mpc']})

omegas = [r['omega'] for r in results]
print(f"\nOmega results ({len(results)} galaxies):")
print(f"  Median: {np.median(omegas):.2f} rad/Gyr")
print(f"  Mean:   {np.mean(omegas):.2f} ± {np.std(omegas):.2f} rad/Gyr")
print(f"\nPublished (Flynn 2026): median = 9.94 rad/Gyr")
print(f"SPARC mean (Flynn & Cannaliato 2025): 7.06 rad/Gyr")
print(f"\nTop 10 by omega:")
for r in sorted(results, key=lambda x: x['omega'], reverse=True)[:10]:
    print(f"  {r['galaxy']:<15} omega={r['omega']:.2f} rad/Gyr")


Omega-ready dwarfs: 24


KeyError: 'Vobs'

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(omegas, bins=15, color='#2ca02c', alpha=0.8, edgecolor='white')
axes[0].axvline(np.median(omegas), color='red', ls='--', lw=1.5,
                label=f'Median={np.median(omegas):.2f}')
axes[0].axvline(7.06, color='blue', ls=':', lw=1.5,
                label='SPARC mean=7.06')
axes[0].set_xlabel(r'$\omega$ (rad/Gyr)', fontsize=11)
axes[0].set_ylabel('N dwarfs', fontsize=11)
axes[0].set_title('Omega Distribution — Dwarf Sample', fontsize=10)
axes[0].legend(fontsize=8)

axes[1].scatter([r['distance'] for r in results], omegas,
                s=40, color='#2ca02c', alpha=0.8)
axes[1].axhline(np.median(omegas), color='red', ls='--', lw=1.2)
axes[1].axhline(7.06, color='blue', ls=':', lw=1.2)
axes[1].set_xlabel('Distance (Mpc)', fontsize=11)
axes[1].set_ylabel(r'$\omega$ (rad/Gyr)', fontsize=11)
axes[1].set_title('Omega vs Distance', fontsize=10)

plt.suptitle('Dwarf Omega Distribution — EPS Research Corpus v1.0\n'
             'Flynn (2026) | DOI: 10.5281/zenodo.20320362', fontsize=10)
plt.tight_layout()
plt.savefig('dw02_omega_distribution.png', dpi=150, bbox_inches='tight')
plt.show()